#### Ingesting Yellow Taxi Trip Data - April 2026

In [63]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *


In [64]:
from pyspark.sql import SparkSession 
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Yellow Taxi Pipeline")
    .config("spark.driver.memory", "4g")
    #.config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

# Reduce verbose logging
spark.sparkContext.setLogLevel("WARN")

print(f"Spark version : {spark.version}")
print(f"App name      : {spark.sparkContext.appName}")
print(f"Master        : {spark.sparkContext.master}")

Spark version : 4.2.0
App name      : Yellow Taxi Pipeline
Master        : local[*]


In [65]:
spark

#### Defining Schema for Yellow Taxi Trip Data

In [66]:
yellow_taxi_schema = '''
                        VendorID INTEGER,
                        tpep_pickup_datetime TIMESTAMP,
                        tpep_dropoff_datetime TIMESTAMP,
                        passenger_count LONG,
                        trip_distance DOUBLE,
                        RatecodeID LONG,
                        store_and_fwd_flag STRING,
                        PULocationID INTEGER,
                        DOLocationID INTEGER,
                        payment_type LONG,
                        fare_amount DOUBLE,
                        extra DOUBLE,
                        mta_tax DOUBLE,
                        tip_amount DOUBLE,
                        tolls_amount DOUBLE,
                        improvement_surcharge DOUBLE,
                        total_amount DOUBLE,
                        congestion_surcharge DOUBLE,
                        airport_fee DOUBLE,
                        cbd_congestion_fee DOUBLE
'''

#### Reading Yellow Taxi Trip Parquet file

In [67]:
yellow_taxi_data = spark.read.option('header',True)\
                                .schema(yellow_taxi_schema)\
                                    .parquet('/app/data/input/yellow/yellow_tripdata_2026-04.parquet')


In [68]:
yellow_taxi_data.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       1| 2026-04-01 00:40:05|  2026-04-01 00:52:44|              1|          2.8|         1|                 N|         237|    

In [69]:
yellow_taxi_data.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [70]:
yellow_taxi_data.rdd.getNumPartitions()

8

#### Data Quality Checks and Filtering

In [71]:
yellow_taxi_data.count()

3831240

##### Checking tip amount > 0 for cash payments --> not a valid case [drop them]

In [72]:

yellow_taxi_data = yellow_taxi_data.\
        filter(~((col('payment_type') == 2) & (col('tip_amount').cast("decimal") > 0.0)))

##### Filtering any row where pick up date is greater than 2026-04

In [73]:
yellow_taxi_data = yellow_taxi_data.where(~(date_format('tpep_pickup_datetime',format = 'yyyy-MM') >='2026-05'))

##### Filtering any row where pick up date is smaller than 2026-04

In [74]:
yellow_taxi_data = yellow_taxi_data.where(~(date_format('tpep_pickup_datetime',format = 'yyyy-MM') <'2026-04'))

##### Filtering rows where passenger count  == 0

In [75]:
yellow_taxi_data = yellow_taxi_data.filter(col('passenger_count') != 0)

##### Filtering rows for negative fare amount

In [76]:
yellow_taxi_data = yellow_taxi_data.filter(col('fare_amount') > 0.0)

##### checking if any trip with negative distance

In [77]:
yellow_taxi_data.filter(col('trip_distance') < 0.0).show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+----

##### Checking if any trip with pickup time later than drop off time

In [78]:
yellow_taxi_data.where(col('tpep_pickup_datetime') > col('tpep_dropoff_datetime')).count()

0

#### Derive trip_duration_minutes

In [79]:

yellow_taxi_data = yellow_taxi_data.withColumn('trip_duration_minutes', timestamp_diff('minute',col('tpep_pickup_datetime'),col('tpep_dropoff_datetime')))

#### Calculating avg_speed_mph

In [80]:
yellow_taxi_data = yellow_taxi_data.withColumn('avg_speed_mph',round(when(col('trip_duration_minutes') == 0, 0).\
                                            otherwise(col('trip_distance')/(col('trip_duration_minutes')/60)),2))

#### Add source file name and ingestion timestamp

In [81]:
yellow_taxi_data.withColumns({'source_file': lit('yellow_taxi_trip_data.parquet'),
                              'ingested_at_timestamp' : current_timestamp()}).show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+---------------------+-------------+--------------------+---------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|cbd_congestion_fee|trip_duration_minutes|avg_speed_mph|         source_file|ingested_at_timestamp|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+----

In [82]:
yellow_taxi_data \
    .withColumn('pickup_date', date_format(col('tpep_pickup_datetime'), 'yyyy-MM-dd')) \
    .write \
    .option('header', True) \
    .partitionBy('pickup_date') \
    .mode('overwrite') \
    .parquet('/app/data/output/yellow')

In [83]:
#spark.stop()
#del spark

#### Read Paritioned Data

In [84]:
read_partitioned_data = spark.read.parquet('/app/data/output/yellow')

In [85]:
read_partitioned_data.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)
 |-- trip_duration_minutes: long (nullable = true)
 |-- avg_speed_mph: double (nulla